In [1]:
import os
import json
import shutil

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

# ==================================================
# STEP 1: CREATE SPARK SESSION
# ==================================================

spark = (
    SparkSession.builder
    .appName("EcommerceRawDataGeneration")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

# ==================================================
# STEP 2: CREATE INPUT DIRECTORY
# ==================================================

input_dir = "input"
os.makedirs(input_dir, exist_ok=True)


# ==================================================
# STEP 3: DEFINE 50 CUSTOMER RECORDS
# ==================================================

customer_data = [
    (101, "Ganesh Kumar", "ganesh.kumar@gmail.com", "Chennai"),
    (102, "Jeyapriya", "jeyapriyak@gmail.com", "Chennai"),
    (103, "Rahul", "rahul19@gmail.com", "Hyderabad"),
    (104, "Swetha", "sneha.mr@gmail.com", "Mumbai"),
    (105, "Karthik Raja", "karthik.raja@gmail.com", "Coimbatore"),
    (106, "Ramya Kumar", "ramya.kumar@gmail.com", "Chennai"),
    (107, "Vijay Anand", "vijay.anand@gmail.com", "Pune"),
    (108, "Meena Devi", "meena.devi@gmail.com", "Madurai"),
    (109, "Suresh Babu", "suresh.babu@gmail.com", "Kochi"),
    (110, "Anitha Suresh", "anitha.suresh@gmail.com", "Delhi"),
    (111, "Ramesh Thilak", "rameshthilak@gmail.com", "Chennai"),
    (112, "Kavya Raj", "kavya.raj@gmail.com", "Kochi"),
    (113, "Vikram Singh", "vikram.singh@gmail.com", "Jaipur"),
    (114, "Lakshmi Devi", "lakshmi.devi@example.com", "Madurai"),
    (115, "Ajay Patel", "ajay.patel@example.com", "Ahmedabad"),
    (116, "Nithya Raj", "nithya.raj@example.com", "Bangalore"),
    (117, "Manoj Kumar", "manoj.kumar@example.com", "Hyderabad"),
    (118, "Swathi Ramesh", "swathi.ramesh@example.com", "Chennai"),
    (119, "Deepak Sharma", "deepak.sharma@example.com", "Delhi"),
    (120, "Pooja Mehta", "pooja.mehta@example.com", "Mumbai"),
    (121, "Harish Babu", "harish.babu@example.com", "Coimbatore"),
    (122, "Revathi S", "revathi.s@example.com", "Chennai"),
    (123, "Gokul Krishna", "gokul.krishna@example.com", "Kochi"),
    (124, "Neha Gupta", "neha.gupta@example.com", "Pune"),
    (125, "Sanjay Rao", "sanjay.rao@example.com", "Hyderabad"),
    (126, "Aishwarya Nair", "aishwarya.nair@example.com", "Kochi"),
    (127, "Prakash M", "prakash.m@example.com", "Chennai"),
    (128, "Monisha Devi", "monisha.devi@example.com", "Madurai"),
    (129, "Tarun Reddy", "tarun.reddy@example.com", "Hyderabad"),
    (130, "Keerthana Raj", "keerthana.raj@example.com", "Coimbatore"),
    (131, "Naveen Kumar", "naveen.kumar@example.com", "Bangalore"),
    (132, "Shalini Das", "shalini.das@example.com", "Kolkata"),
    (133, "Abhishek Jain", "abhishek.jain@example.com", "Jaipur"),
    (134, "Geetha Lakshmi", "geetha.lakshmi@example.com", "Chennai"),
    (135, "Surya Prakash", "surya.prakash@example.com", "Hyderabad"),
    (136, "Rohini Menon", "rohini.menon@example.com", "Kochi"),
    (137, "Dinesh Raj", "dinesh.raj@gmail.com", "Madurai"),
    (138, "Preethi Suresh", "preethi.suresh@gmail.com", "Bangalore"),
    (139, "Mohan Das", "mohan.das@gmail.com", "Kolkata"),
    (140, "Yamini Rao", "yamini.rao@gmail.com", "Mumbai"),
    (141, "Sathish Kumar", "sathish.kumar@gmail.com", "Chennai"),
    (142, "Anjali Verma", "anjali.verma@gmail.com", "Delhi"),
    (143, "Girish Nair", "girish.nair@gmail.com", "Kochi"),
    (144, "Ritika Shah", "ritika.shah@gmail.com", "Ahmedabad"),
    (145, "Bharath Raj", "bharath.raj@gmail.com", "Coimbatore"),
    (146, "Janani Kumar", "janani.kumar@gmail.com", "Chennai"),
    (147, "Varun Mehta", "varun.mehta@gmail.com", "Pune"),
    (148, "Sindhu R", "sindhu.r@gmail.com", "Bangalore"),
    (149, "Aditya Sanjay", "aditya.sanjay@gmail.com", "Hyderabad"),
    (150, "Mahalakshmi S", "mahalakshmi.s@gmail.com", "Madurai")
]

# ==================================================
# STEP 4: DEFINE CUSTOMER SCHEMA
# ==================================================

customer_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), False),
    StructField("email", StringType(), False),
    StructField("city", StringType(), False)
])

# ==================================================
# STEP 5: CREATE CUSTOMER DATAFRAME
# ==================================================

customers_df = spark.createDataFrame(
    customer_data,
    schema=customer_schema
)

# Validate customer records
customer_count = customers_df.count()

assert customer_count == 50, (
    f"Expected 50 customers, but found {customer_count}"
)

# ==================================================
# STEP 6: SAVE CUSTOMERS.CSV
# ==================================================

customer_csv_temp = os.path.join(
    input_dir, "customers_csv_temp"
)

(
    customers_df.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(customer_csv_temp)
)

customer_csv_part = next(
    filename
    for filename in os.listdir(customer_csv_temp)
    if filename.startswith("part-") and filename.endswith(".csv")
)

shutil.move(
    os.path.join(customer_csv_temp, customer_csv_part),
    os.path.join(input_dir, "customers.csv")
)

shutil.rmtree(customer_csv_temp)

# ==================================================
# STEP 7: SAVE CUSTOMER_PROFILES.JSON
# ==================================================

customer_profiles = [
    {
        "customer_id": customer_id,
        "customer_name": customer_name,
        "email": email,
        "city": city
    }
    for customer_id, customer_name, email, city in customer_data
]

with open(
    os.path.join(input_dir, "customer_profiles.json"),
    "w",
    encoding="utf-8"
) as json_file:
    json.dump(customer_profiles, json_file, indent=4)


# ==================================================
# STEP 8: GENERATE 147 UNIQUE ORDER RECORDS
# ==================================================

order_data = []

for order_id in range(1001, 1148):

    # Assign customer IDs between 101 and 150
    customer_id = 101 + ((order_id - 1001) % 50)

    # Intentionally reference a nonexistent customer
    if order_id == 1147:
        customer_id = 151

    # Generate an amount as a string so invalid raw
    # values can be preserved for later cleaning.
    order_amount = str(
        float(100 + ((order_id - 1001) % 50) * 25)
    )

    # Intentionally introduce data-quality issues
    if order_id == 1005:
        order_amount = None

    elif order_id == 1007:
        order_amount = "INVALID"

    elif order_id == 1015:
        order_amount = None

    order_data.append(
        (order_id, customer_id, order_amount)
    )


# ==================================================
# STEP 9: ADD THREE DUPLICATE ORDER RECORDS
# ==================================================

for duplicate_order_id in [1001, 1010, 1020]:

    duplicate_record = next(
        record
        for record in order_data
        if record[0] == duplicate_order_id
    )

    order_data.append(duplicate_record)


# ==================================================
# STEP 10: DEFINE ORDER SCHEMA
# ==================================================

order_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_amount", StringType(), True)
])


# ==================================================
# STEP 11: CREATE ORDERS DATAFRAME
# ==================================================

orders_df = spark.createDataFrame(
    order_data,
    schema=order_schema
)

# ==================================================
# STEP 12: SAVE ORDERS.CSV
# ==================================================

orders_csv_temp = os.path.join(
    input_dir, "orders_csv_temp"
)

(
    orders_df.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(orders_csv_temp)
)

orders_csv_part = next(
    filename
    for filename in os.listdir(orders_csv_temp)
    if filename.startswith("part-") and filename.endswith(".csv")
)

shutil.move(
    os.path.join(orders_csv_temp, orders_csv_part),
    os.path.join(input_dir, "orders.csv")
)

shutil.rmtree(orders_csv_temp)


# ==================================================
# STEP 13: VALIDATE GENERATED DATA
# ==================================================

raw_order_count = orders_df.count()

unique_order_count = (
    orders_df
    .select("order_id")
    .distinct()
    .count()
)

duplicate_order_count = raw_order_count - unique_order_count

assert raw_order_count == 150
assert unique_order_count == 147
assert duplicate_order_count == 3

# Verify duplicate IDs
duplicate_ids = [
    row["order_id"]
    for row in (
        orders_df.groupBy("order_id")
        .count()
        .filter("count > 1")
        .orderBy("order_id")
        .collect()
    )
]

assert duplicate_ids == [1001, 1010, 1020]

# Verify invalid or missing amounts
invalid_amount_count = (
    orders_df
    .filter(
        "order_amount IS NULL OR order_amount = 'INVALID'"
    )
    .count()
)

assert invalid_amount_count == 3

# Verify missing customer reference
invalid_customer_count = (
    orders_df
    .filter("customer_id = 151")
    .count()
)

assert invalid_customer_count == 1

# ==================================================
# STEP 14: DISPLAY FINAL RESULTS
# ==================================================

print("Source datasets created.")
print()
print(f"Customer rows: {customer_count}")
print(f"Raw order rows: {raw_order_count}")
print(f"Unique order IDs before cleaning: {unique_order_count}")

print(
    "Input files:",
    sorted([
        "customer_profiles.json",
        "customers.csv",
        "orders.csv"
    ])
)

print()
print("Duplicate order IDs:", duplicate_ids)
print("Orders with invalid/missing amounts:", invalid_amount_count)
print("Orders with missing customer references:", invalid_customer_count)

print("\nCustomer sample:")
customers_df.show(5, truncate=False)

print("\nOrder sample:")
orders_df.show(5, truncate=False)

# ==================================================
# STEP 15: STOP SPARK SESSION
# ==================================================

spark.stop()


Source datasets created.

Customer rows: 50
Raw order rows: 150
Unique order IDs before cleaning: 147
Input files: ['customer_profiles.json', 'customers.csv', 'orders.csv']

Duplicate order IDs: [1001, 1010, 1020]
Orders with invalid/missing amounts: 3
Orders with missing customer references: 1

Customer sample:
+-----------+-------------+----------------------+----------+
|customer_id|customer_name|email                 |city      |
+-----------+-------------+----------------------+----------+
|101        |Ganesh Kumar |ganesh.kumar@gmail.com|Chennai   |
|102        |Jeyapriya    |jeyapriyak@gmail.com  |Chennai   |
|103        |Rahul        |rahul19@gmail.com     |Hyderabad |
|104        |Swetha       |sneha.mr@gmail.com    |Mumbai    |
|105        |Karthik Raja |karthik.raja@gmail.com|Coimbatore|
+-----------+-------------+----------------------+----------+
only showing top 5 rows

Order sample:
+--------+-----------+------------+
|order_id|customer_id|order_amount|
+--------+-------

In [2]:
# Step 4: Recreate Spark Session and Read CSV Files into Spark DataFrames

from pyspark.sql import SparkSession

# Recreate the Spark session if it was stopped
spark = (
    SparkSession.builder
    .appName("EcommerceDataLoading")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# Define file paths
customers_path = "/content/input/customers.csv"
orders_path = "/content/input/orders.csv"

# Read customers CSV
customers_df = spark.read.csv(
    customers_path,
    header=True,
    inferSchema=True
)

# Read orders CSV
orders_df = spark.read.csv(
    orders_path,
    header=True,
    inferSchema=True
)

# Display first 10 customer records
print("Customer Data:")
customers_df.show(10, truncate=False)

# Display first 10 order records
print("Order Data:")
orders_df.show(10, truncate=False)

# Check row counts
print("Customer Row Count:", customers_df.count())
print("Order Row Count:", orders_df.count())

# Display schemas
print("Customer Schema:")
customers_df.printSchema()

print("Order Schema:")
orders_df.printSchema()

Customer Data:
+-----------+-------------+-----------------------+----------+
|customer_id|customer_name|email                  |city      |
+-----------+-------------+-----------------------+----------+
|101        |Ganesh Kumar |ganesh.kumar@gmail.com |Chennai   |
|102        |Jeyapriya    |jeyapriyak@gmail.com   |Chennai   |
|103        |Rahul        |rahul19@gmail.com      |Hyderabad |
|104        |Swetha       |sneha.mr@gmail.com     |Mumbai    |
|105        |Karthik Raja |karthik.raja@gmail.com |Coimbatore|
|106        |Ramya Kumar  |ramya.kumar@gmail.com  |Chennai   |
|107        |Vijay Anand  |vijay.anand@gmail.com  |Pune      |
|108        |Meena Devi   |meena.devi@gmail.com   |Madurai   |
|109        |Suresh Babu  |suresh.babu@gmail.com  |Kochi     |
|110        |Anitha Suresh|anitha.suresh@gmail.com|Delhi     |
+-----------+-------------+-----------------------+----------+
only showing top 10 rows
Order Data:
+--------+-----------+------------+
|order_id|customer_id|order_am

In [3]:
from pyspark.sql.functions import col, expr

# Re-read orders CSV with inferSchema=False to treat all columns as strings first
orders_df = spark.read.csv(
    "/content/input/orders.csv",
    header=True,
    inferSchema=False
)

# Cast the ID columns to integer
orders_df = orders_df.withColumn("order_id", col("order_id").cast("integer"))
orders_df = orders_df.withColumn("customer_id", col("customer_id").cast("integer"))

print("Original Orders Schema (before casting amount):")
orders_df.printSchema()

# Safely convert order_amount to double using try_cast to handle 'INVALID' gracefully as NULL
orders_df = orders_df.withColumn(
    "order_amount",
    expr("try_cast(order_amount as double)")
)

print("Updated Orders Data:")
orders_df.show(10, truncate=False)

print("Updated Orders Schema:")
orders_df.printSchema()

Original Orders Schema (before casting amount):
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_amount: string (nullable = true)

Updated Orders Data:
+--------+-----------+------------+
|order_id|customer_id|order_amount|
+--------+-----------+------------+
|1001    |101        |100.0       |
|1002    |102        |125.0       |
|1003    |103        |150.0       |
|1004    |104        |175.0       |
|1005    |105        |NULL        |
|1006    |106        |225.0       |
|1007    |107        |NULL        |
|1008    |108        |275.0       |
|1009    |109        |300.0       |
|1010    |110        |325.0       |
+--------+-----------+------------+
only showing top 10 rows
Updated Orders Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_amount: double (nullable = true)



In [4]:

# Step 5B: Remove duplicate order IDs

# 1. Count records before removing duplicates
before_count = orders_df.count()

print("Before removing duplicates:", before_count)

# 2. Remove duplicate order IDs
orders_df = orders_df.dropDuplicates(["order_id"])

# 3. Count records after removing duplicates
after_count = orders_df.count()

print("After removing duplicates:", after_count)

# 4. Calculate the number of duplicate records removed
duplicates_removed = before_count - after_count

print("Duplicate records removed:", duplicates_removed)


Before removing duplicates: 150
After removing duplicates: 147
Duplicate records removed: 3


In [5]:
from pyspark.sql.functions import col, when, sum as spark_sum

# Step 1: Inspect records with invalid or missing amounts
invalid_amounts_df = orders_df.filter(
    col("order_amount").isNull()
)

# Step 2: Count invalid or missing amount records
invalid_count = invalid_amounts_df.count()

print("Invalid or missing amount records:", invalid_count)

# Step 3: Display invalid records for inspection
print("Records with invalid or missing amounts:")
invalid_amounts_df.show(truncate=False)

# Step 4: Remove records with invalid or missing amounts
orders_df = orders_df.filter(
    col("order_amount").isNotNull()
)

# Step 5: Count valid orders after cleaning
valid_count = orders_df.count()

print("Valid orders after cleaning:", valid_count)

# Step 6: Validate that no missing amounts remain
remaining_invalid = orders_df.filter(
    col("order_amount").isNull()
).count()

print("Remaining invalid or missing amounts:", remaining_invalid)

Invalid or missing amount records: 3
Records with invalid or missing amounts:
+--------+-----------+------------+
|order_id|customer_id|order_amount|
+--------+-----------+------------+
|1005    |105        |NULL        |
|1015    |115        |NULL        |
|1007    |107        |NULL        |
+--------+-----------+------------+

Valid orders after cleaning: 144
Remaining invalid or missing amounts: 0


In [6]:
from pyspark.sql.functions import col, lit, when, round, expr

# Reset and reload a clean copy of orders_df from the input source
orders_df = spark.read.csv(
    "/content/input/orders.csv",
    header=True,
    inferSchema=False
)

# Step 1: Safely convert order_amount into DOUBLE
orders_df = orders_df.withColumn(
    "order_amount",
    expr("try_cast(order_amount AS DOUBLE)")
)

# Step 1B: Drop duplicate order records before identifying valid/invalid records
orders_df = orders_df.dropDuplicates(["order_id"])

# Step 2: Separate valid records (should now be 144 records)
valid_orders_df = orders_df.filter(
    col("order_amount").isNotNull()
)

# Step 3: Separate invalid records
invalid_orders_df = orders_df.filter(
    col("order_amount").isNull()
)

# Step 4: Calculate 5% service charge
valid_orders_df = valid_orders_df.withColumn(
    "service_charge",
    round(col("order_amount") * 0.05, 2)
)

# Step 5: Calculate total amount
valid_orders_df = valid_orders_df.withColumn(
    "total_amount",
    round(col("order_amount") + col("service_charge"), 2)
)

# Step 6: Add currency
valid_orders_df = valid_orders_df.withColumn(
    "currency",
    lit("INR")
)

# Step 7: Categorize orders
valid_orders_df = valid_orders_df.withColumn(
    "order_category",
    when(col("order_amount") >= 20000.0, "High")
    .otherwise("Normal")
)

# Step 8: Display results
valid_orders_df.show(10, truncate=False)

+--------+-----------+------------+--------------+------------+--------+--------------+
|order_id|customer_id|order_amount|service_charge|total_amount|currency|order_category|
+--------+-----------+------------+--------------+------------+--------+--------------+
|1001    |101        |100.0       |5.0           |105.0       |INR     |Normal        |
|1002    |102        |125.0       |6.25          |131.25      |INR     |Normal        |
|1003    |103        |150.0       |7.5           |157.5       |INR     |Normal        |
|1004    |104        |175.0       |8.75          |183.75      |INR     |Normal        |
|1006    |106        |225.0       |11.25         |236.25      |INR     |Normal        |
|1008    |108        |275.0       |13.75         |288.75      |INR     |Normal        |
|1009    |109        |300.0       |15.0          |315.0       |INR     |Normal        |
|1010    |110        |325.0       |16.25         |341.25      |INR     |Normal        |
|1011    |111        |350.0     

In [7]:
from pyspark.sql.functions import col, count

# --------------------------------------------------
# STEP 1: Display customer data
# --------------------------------------------------

print("Customer Data:")
customers_df.show(10, truncate=False)

print("Customer Schema:")
customers_df.printSchema()


# --------------------------------------------------
# STEP 2: Check duplicate customer IDs
# --------------------------------------------------

duplicate_customers_df = (
    customers_df
    .groupBy("customer_id")
    .agg(count("*").alias("record_count"))
    .filter(col("record_count") > 1)
)

print("Duplicate Customer IDs:")
duplicate_customers_df.show()

print("Duplicate Customer Count:", duplicate_customers_df.count())


# --------------------------------------------------
# STEP 3: Perform LEFT JOIN
# --------------------------------------------------

joined_orders_df = valid_orders_df.join(
    customers_df.select(
        "customer_id",
        "customer_name",
        "city"
    ),
    on="customer_id",
    how="left"
)

print("Orders Joined with Customer Details:")
joined_orders_df.show(10, truncate=False)


# --------------------------------------------------
# STEP 4: Validate record counts
# --------------------------------------------------

valid_order_count = valid_orders_df.count()

joined_order_count = joined_orders_df.count()

print("Valid Orders Count:", valid_order_count)
print("Joined Orders Count:", joined_order_count)

if valid_order_count == joined_order_count:
    print("PASS: All valid orders are preserved.")
else:
    print("FAIL: Record count mismatch detected.")


# --------------------------------------------------
# STEP 5: Identify unmatched customer records
# --------------------------------------------------

unmatched_orders_df = joined_orders_df.filter(
    col("customer_name").isNull()
)

print("Unmatched Customer Orders:")
unmatched_orders_df.show(truncate=False)

print("Unmatched Orders Count:", unmatched_orders_df.count())


# --------------------------------------------------
# STEP 6: Validate Order ID 1147
# --------------------------------------------------

order_1147_df = joined_orders_df.filter(
    col("order_id") == "1147"
)

print("Validation for Order ID 1147:")

order_1147_df.select(
    "order_id",
    "customer_id",
    "customer_name",
    "city"
).show(truncate=False)


# --------------------------------------------------
# STEP 7: Final validation
# --------------------------------------------------

print("========== STEP 7 SUMMARY ==========")

print("Valid Orders:", valid_order_count)

print("Joined Orders:", joined_order_count)

print("Duplicate Customer IDs:", duplicate_customers_df.count())

print("Unmatched Customer Orders:", unmatched_orders_df.count())

print("====================================")

Customer Data:
+-----------+-------------+-----------------------+----------+
|customer_id|customer_name|email                  |city      |
+-----------+-------------+-----------------------+----------+
|101        |Ganesh Kumar |ganesh.kumar@gmail.com |Chennai   |
|102        |Jeyapriya    |jeyapriyak@gmail.com   |Chennai   |
|103        |Rahul        |rahul19@gmail.com      |Hyderabad |
|104        |Swetha       |sneha.mr@gmail.com     |Mumbai    |
|105        |Karthik Raja |karthik.raja@gmail.com |Coimbatore|
|106        |Ramya Kumar  |ramya.kumar@gmail.com  |Chennai   |
|107        |Vijay Anand  |vijay.anand@gmail.com  |Pune      |
|108        |Meena Devi   |meena.devi@gmail.com   |Madurai   |
|109        |Suresh Babu  |suresh.babu@gmail.com  |Kochi     |
|110        |Anitha Suresh|anitha.suresh@gmail.com|Delhi     |
+-----------+-------------+-----------------------+----------+
only showing top 10 rows
Customer Schema:
root
 |-- customer_id: integer (nullable = true)
 |-- custome

In [8]:
from pyspark.sql.functions import sum, count

# Define category_sales_df by grouping joined_orders_df by category and counting orders
category_sales_df = (
    joined_orders_df
    .groupBy("order_category")
    .agg(count("order_id").alias("order_count"))
)

total_orders = joined_orders_df.count()

aggregated_orders = category_sales_df.agg(
    sum("order_count").alias("total")
).collect()[0]["total"]

print("Original Orders:", total_orders)
print("Aggregated Orders:", aggregated_orders)

if total_orders == aggregated_orders:
    print("PASS: Category-wise aggregation is successful.")
else:
    print("FAIL: Order count mismatch.")

Original Orders: 144
Aggregated Orders: 144
PASS: Category-wise aggregation is successful.


In [9]:
from pyspark.sql.functions import sum, avg, count, round, col

# City-wise sales aggregation
city_sales_df = joined_orders_df.groupBy("city").agg(
    round(sum("order_amount"), 2).alias("total_sales"),
    round(avg("order_amount"), 2).alias("average_order_amount"),
    count("order_id").alias("order_count")
)

# Display results
print("========== CITY-WISE SALES SUMMARY ==========")

city_sales_df.orderBy(
    col("total_sales").desc_nulls_last()
).show(truncate=False)

========== CITY-WISE SALES SUMMARY ==========
+----------+-----------+--------------------+-----------+
|city      |total_sales|average_order_amount|order_count|
+----------+-----------+--------------------+-----------+
|Chennai   |17850.0    |595.0               |30         |
|Kochi     |12525.0    |695.83              |18         |
|Hyderabad |11900.0    |700.0               |17         |
|Madurai   |10075.0    |719.64              |14         |
|Bangalore |9600.0     |872.73              |11         |
|Coimbatore|8275.0     |752.27              |11         |
|Delhi     |6000.0     |666.67              |9          |
|Kolkata   |5775.0     |962.5               |6          |
|Mumbai    |5475.0     |608.33              |9          |
|Pune      |5025.0     |717.86              |7          |
|Ahmedabad |4425.0     |885.0               |5          |
|Jaipur    |3900.0     |650.0               |6          |
|NULL      |1250.0     |1250.0              |1          |
+----------+-----------+--

In [10]:
from pyspark.sql.functions import sum, count, round, col

# Calculate overall sales and valid order count
overall_sales_df = valid_orders_df.agg(
    round(sum("order_amount"), 2).alias("total_sales"),
    count("order_id").alias("valid_order_count")
)

print("========== OVERALL SALES VALIDATION ==========")

overall_sales_df.show(truncate=False)

# Extract summary values
sales_summary = overall_sales_df.collect()[0]

total_sales = sales_summary["total_sales"]
valid_order_count = sales_summary["valid_order_count"]

print("Total Sales:", total_sales)
print("Valid Order Count:", valid_order_count)

# Validate order count
if valid_order_count == 144:
    print("PASS: Valid order count is 144.")
else:
    print("FAIL: Expected 144 valid orders, but found", valid_order_count)

# Validate sales total against joined orders using Python's built-in round function
raw_joined_sales = joined_orders_df.agg(sum("order_amount")).collect()[0][0]
joined_sales = __builtins__.round(raw_joined_sales, 2) if raw_joined_sales is not None else 0.0

if total_sales == joined_sales:
    print("PASS: Overall sales total matches joined orders.")
else:
    print("FAIL: Overall sales total mismatch.")

print("===============================================")

========== OVERALL SALES VALIDATION ==========
+-----------+-----------------+
|total_sales|valid_order_count|
+-----------+-----------------+
|102075.0   |144              |
+-----------+-----------------+

Total Sales: 102075.0
Valid Order Count: 144
PASS: Valid order count is 144.
PASS: Overall sales total matches joined orders.


In [11]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# Define window specification based on order_id descending (since order_date does not exist)
customer_window = Window.partitionBy(
    "customer_id"
).orderBy(
    col("order_id").desc()
)

# Assign row numbers
customer_orders_df = joined_orders_df.withColumn(
    "row_number",
    row_number().over(customer_window)
)

# Display all orders sorted by customer and row_number
print("========== CUSTOMER LATEST ORDER DETAILS ==========")

customer_orders_df.orderBy(
    "customer_id",
    "row_number"
).show(50, truncate=False)

========== CUSTOMER LATEST ORDER DETAILS ==========
+-----------+--------+------------+--------------+------------+--------+--------------+-------------+----------+----------+
|customer_id|order_id|order_amount|service_charge|total_amount|currency|order_category|customer_name|city      |row_number|
+-----------+--------+------------+--------------+------------+--------+--------------+-------------+----------+----------+
|101        |1101    |100.0       |5.0           |105.0       |INR     |Normal        |Ganesh Kumar |Chennai   |1         |
|101        |1051    |100.0       |5.0           |105.0       |INR     |Normal        |Ganesh Kumar |Chennai   |2         |
|101        |1001    |100.0       |5.0           |105.0       |INR     |Normal        |Ganesh Kumar |Chennai   |3         |
|102        |1102    |125.0       |6.25          |131.25      |INR     |Normal        |Jeyapriya    |Chennai   |1         |
|102        |1052    |125.0       |6.25          |131.25      |INR     |Normal  

In [12]:
from pyspark.sql.functions import col, sum, round

# Cache the joined DataFrame
cached_orders_df = joined_orders_df.cache()

# Trigger execution and populate cache
cached_orders_count = cached_orders_df.count()

print("========== CACHE VALIDATION ==========")

print("Valid Joined Orders:", cached_orders_count)
print("Is Cached:", cached_orders_df.is_cached)

if cached_orders_count == 144 and cached_orders_df.is_cached:
    print("PASS: DataFrame cached successfully.")
else:
    print("FAIL: Check order count or cache status.")

========== CACHE VALIDATION ==========
Valid Joined Orders: 144
Is Cached: True
PASS: DataFrame cached successfully.


In [13]:
cached_sales = cached_orders_df.agg(
    round(sum("order_amount"), 2).alias("total_sales")
).collect()[0]["total_sales"]

print("Step 8 Total Sales:", total_sales)
print("Cached DataFrame Sales:", cached_sales)

if total_sales == cached_sales:
    print("PASS: Total sales match Step 8.")
else:
    print("FAIL: Sales total mismatch.")

Step 8 Total Sales: 102075.0
Cached DataFrame Sales: 102075.0
PASS: Total sales match Step 8.


In [14]:
high_value_orders_df = cached_orders_df.filter(
    col("order_amount") >= 20000
)

print("========== HIGH VALUE ORDERS ==========")

high_value_orders_df.show(20, truncate=False)

print("High Value Orders Count:", high_value_orders_df.count())

========== HIGH VALUE ORDERS ==========
+-----------+--------+------------+--------------+------------+--------+--------------+-------------+----+
|customer_id|order_id|order_amount|service_charge|total_amount|currency|order_category|customer_name|city|
+-----------+--------+------------+--------------+------------+--------+--------------+-------------+----+
+-----------+--------+------------+--------------+------------+--------+--------------+-------------+----+

High Value Orders Count: 0


In [15]:
cached_orders_df.unpersist()

print("Is Cached After Unpersist:", cached_orders_df.is_cached)


Is Cached After Unpersist: False


In [16]:
from pyspark import StorageLevel

persisted_orders_df = joined_orders_df.persist(
    StorageLevel.MEMORY_AND_DISK
)

persisted_count = persisted_orders_df.count()

print("Persisted Orders:", persisted_count)
print("Is Persisted:", persisted_orders_df.is_cached)

# Release persisted data
persisted_orders_df.unpersist()

Persisted Orders: 144
Is Persisted: True


DataFrame[customer_id: string, order_id: string, order_amount: double, service_charge: double, total_amount: double, currency: string, order_category: string, customer_name: string, city: string]

In [17]:
# Check initial partition count
initial_partitions = joined_orders_df.rdd.getNumPartitions()

print("Initial Partitions:", initial_partitions)
# Increase partitions to 4
repartitioned_df = joined_orders_df.repartition(4)

print("After repartition:", repartitioned_df.rdd.getNumPartitions())
# Reduce partitions to 1
coalesced_df = repartitioned_df.coalesce(1)

print("After coalesce:", coalesced_df.rdd.getNumPartitions())
print("========== PARTITION VALIDATION ==========")

print("Initial Partitions:", initial_partitions)
print("After repartition:", repartitioned_df.rdd.getNumPartitions())
print("After coalesce:", coalesced_df.rdd.getNumPartitions())

if repartitioned_df.rdd.getNumPartitions() == 4:
    print("PASS: Repartition successful.")
else:
    print("FAIL: Repartition unsuccessful.")

if coalesced_df.rdd.getNumPartitions() == 1:
    print("PASS: Coalesce successful.")
else:
    print("FAIL: Coalesce unsuccessful.")

Initial Partitions: 1
After repartition: 4
After coalesce: 1
========== PARTITION VALIDATION ==========
Initial Partitions: 1
After repartition: 4
After coalesce: 1
PASS: Repartition successful.
PASS: Coalesce successful.


In [19]:
from pyspark.sql.functions import col, explode_outer

# Load the customer profiles JSON file into profiles_df
profiles_df = spark.read.option("multiLine", "true").json("/content/input/customer_profiles.json")

# Flatten nested address fields and explode skills array
# Note: Since address and skills are not defined in the basic profiles JSON generated in cell 8DNpwLLgv2-d,
# we will load the schema safely and use select
flattened_profiles_df = profiles_df.select(
    col("customer_id"),
    col("customer_name"),
    col("city")
)

# Display flattened data
print("========== FLATTENED CUSTOMER PROFILES ==========")
flattened_profiles_df.show(truncate=False)

# Display schema
print("========== FLATTENED DATAFRAME SCHEMA ==========")
flattened_profiles_df.printSchema()

========== FLATTENED CUSTOMER PROFILES ==========
+-----------+-------------+----------+
|customer_id|customer_name|city      |
+-----------+-------------+----------+
|101        |Ganesh Kumar |Chennai   |
|102        |Jeyapriya    |Chennai   |
|103        |Rahul        |Hyderabad |
|104        |Swetha       |Mumbai    |
|105        |Karthik Raja |Coimbatore|
|106        |Ramya Kumar  |Chennai   |
|107        |Vijay Anand  |Pune      |
|108        |Meena Devi   |Madurai   |
|109        |Suresh Babu  |Kochi     |
|110        |Anitha Suresh|Delhi     |
|111        |Ramesh Thilak|Chennai   |
|112        |Kavya Raj    |Kochi     |
|113        |Vikram Singh |Jaipur    |
|114        |Lakshmi Devi |Madurai   |
|115        |Ajay Patel   |Ahmedabad |
|116        |Nithya Raj   |Bangalore |
|117        |Manoj Kumar  |Hyderabad |
|118        |Swathi Ramesh|Chennai   |
|119        |Deepak Sharma|Delhi     |
|120        |Pooja Mehta  |Mumbai    |
+-----------+-------------+----------+
only showing t

In [20]:
#Step 13A – Save final order-level dataset as Parquet
# Define output paths
parquet_path = "/content/output/final_orders_parquet"

# Save final orders as Parquet
joined_orders_df.write \
    .mode("overwrite") \
    .parquet(parquet_path)

print("PASS: Final orders saved as Parquet.")

PASS: Final orders saved as Parquet.


In [21]:
#Step 13B – Save category report as CSV
csv_path = "/content/output/category_sales_csv"

category_sales_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(csv_path)

print("PASS: Category sales report saved as CSV.")
#Step 13C – Save city report as JSON
json_path = "/content/output/city_sales_json"

city_sales_df.write \
    .mode("overwrite") \
    .json(json_path)

print("PASS: City sales report saved as JSON.")
#Step 13D – Inspect output folders
import os

output_path = "/content/output"

print("========== OUTPUT FOLDER INSPECTION ==========")

for root, directories, files in os.walk(output_path):
    print("\nFolder:", root)

    for file in files:
        print("File:", file)

print("\n===============================================")
#Step 13E – Read back saved files
#Read Parquet
saved_orders_df = spark.read.parquet(parquet_path)

print("========== SAVED PARQUET DATA ==========")

saved_orders_df.show(10, truncate=False)

print("Saved Parquet row count:", saved_orders_df.count())
#Read CSV
saved_category_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(csv_path)

print("========== SAVED CATEGORY CSV ==========")

saved_category_df.show(truncate=False)

print("Saved Category Report Count:", saved_category_df.count())
#Read JSON
saved_city_df = spark.read.json(json_path)

print("========== SAVED CITY JSON ==========")

saved_city_df.show(truncate=False)

print("Saved City Report Count:", saved_city_df.count())


PASS: Category sales report saved as CSV.
PASS: City sales report saved as JSON.
========== OUTPUT FOLDER INSPECTION ==========

Folder: /content/output

Folder: /content/output/category_sales_csv
File: part-00000-7893d530-7d81-42f9-8652-9871dbbed405-c000.csv
File: ._SUCCESS.crc
File: .part-00000-7893d530-7d81-42f9-8652-9871dbbed405-c000.csv.crc
File: _SUCCESS

Folder: /content/output/city_sales_json
File: ._SUCCESS.crc
File: .part-00000-35c43ba0-a832-40b5-9934-fae684cc907e-c000.json.crc
File: _SUCCESS
File: part-00000-35c43ba0-a832-40b5-9934-fae684cc907e-c000.json

Folder: /content/output/final_orders_parquet
File: ._SUCCESS.crc
File: part-00000-7aa61be4-cd45-420f-af2b-d6072f8f131f-c000.snappy.parquet
File: _SUCCESS
File: .part-00000-7aa61be4-cd45-420f-af2b-d6072f8f131f-c000.snappy.parquet.crc

========== SAVED PARQUET DATA ==========
+-----------+--------+------------+--------------+------------+--------+--------------+-------------+---------+
|customer_id|order_id|order_amount|servi

In [25]:
#Step -14 - Final data-quality validation

# Load the raw dataset to perform complete validation checks
orders_raw_df = spark.read.csv(
    "/content/input/orders.csv",
    header=True,
    inferSchema=False
)

orders_deduplicated_df = orders_raw_df.dropDuplicates(["order_id"])

#Validate record counts
print("========== RECORD COUNT VALIDATION ==========")

raw_count = orders_raw_df.count()
deduplicated_count = orders_deduplicated_df.count()
invalid_count = invalid_orders_df.count()
final_count = valid_orders_df.count()

print("Raw Order Rows:", raw_count)
print("After Duplicate Removal:", deduplicated_count)
print("Invalid/Missing Amount Records:", invalid_count)
print("Final Valid Orders:", final_count)

print("\n========== VALIDATION RESULTS ==========")

checks = {
    "Raw Orders": (raw_count, 150),
    "After Deduplication": (deduplicated_count, 147),
    "Invalid Amount Records": (invalid_count, 3),
    "Final Valid Orders": (final_count, 144)
}
# Validate unique order IDs
from pyspark.sql.functions import count, col

total_order_ids = valid_orders_df.select("order_id").count()

unique_order_ids = valid_orders_df.select(
    "order_id"
).distinct().count()

duplicate_order_ids = (
    valid_orders_df.groupBy("order_id")
    .agg(count("*").alias("record_count"))
    .filter(col("record_count") > 1)
)

print("Total Order IDs:", total_order_ids)
print("Unique Order IDs:", unique_order_ids)
print("Duplicate Order IDs:", duplicate_order_ids.count())

if total_order_ids == unique_order_ids == 144:
    print("PASS: All final order IDs are unique.")
else:
    print("FAIL: Duplicate or missing order IDs detected.")
#Validate NULL amounts
null_amount_count = valid_orders_df.filter(
    col("order_amount").isNull()
).count()

print("Final NULL Amounts:", null_amount_count)

if null_amount_count == 0:
    print("PASS: No NULL amounts found.")
else:
    print("FAIL: NULL amounts detected.")
#Validate overall sales amount
from pyspark.sql.functions import sum, round

final_sales_total = valid_orders_df.agg(
    round(sum("order_amount"), 2).alias("total_sales")
).collect()[0]["total_sales"]

print("Final Sales Total:", final_sales_total)
print("Step 8 Sales Total:", total_sales)
print("Saved Parquet Sales Total:",
      saved_orders_df.agg(
          round(sum("order_amount"), 2)
      ).collect()[0][0])

if final_sales_total == total_sales:
    print("PASS: Final sales match Step 8.")
else:
    print("FAIL: Sales total mismatch.")

saved_parquet_sales = saved_orders_df.agg(
    round(sum("order_amount"), 2)
).collect()[0][0]

if final_sales_total == saved_parquet_sales:
    print("PASS: Saved Parquet sales match final sales.")
else:
    print("FAIL: Saved Parquet sales mismatch.")

print("\n========== FINAL ETL DATA QUALITY REPORT ==========")
#Final ETL quality report
print("Raw Orders:", raw_count)
print("Deduplicated Orders:", deduplicated_count)
print("Invalid Orders:", invalid_count)
print("Final Valid Orders:", final_count)
print("Unique Order IDs:", unique_order_ids)
print("NULL Amounts:", null_amount_count)
print("Final Sales Total:", final_sales_total)

print("===================================================")

========== RECORD COUNT VALIDATION ==========
Raw Order Rows: 150
After Duplicate Removal: 147
Invalid/Missing Amount Records: 3
Final Valid Orders: 144

========== VALIDATION RESULTS ==========
Total Order IDs: 144
Unique Order IDs: 144
Duplicate Order IDs: 0
PASS: All final order IDs are unique.
Final NULL Amounts: 0
PASS: No NULL amounts found.
Final Sales Total: 102075.0
Step 8 Sales Total: 102075.0
Saved Parquet Sales Total: 102075.0
PASS: Final sales match Step 8.
PASS: Saved Parquet sales match final sales.

========== FINAL ETL DATA QUALITY REPORT ==========
Raw Orders: 150
Deduplicated Orders: 147
Invalid Orders: 3
Final Valid Orders: 144
Unique Order IDs: 144
NULL Amounts: 0
Final Sales Total: 102075.0


In [27]:
#Step 15 — Malformed-data handling modes
# CSV file path
csv_file_path = "/content/input/orders.csv"

# Read using PERMISSIVE mode
permissive_df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .option("mode", "PERMISSIVE") \
    .csv(csv_file_path)

print("Rows read in PERMISSIVE mode:", permissive_df.count())

permissive_df.show(10, truncate=False)
# Read CSV in DROPMALFORMED mode
dropmalformed_df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .option("mode", "DROPMALFORMED") \
    .csv(csv_file_path)

print("Rows read in DROPMALFORMED mode:", dropmalformed_df.count())
#Read CSV in FAILFAST mode
failfast_df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .option("mode", "FAILFAST") \
    .csv(csv_file_path)

print("Rows read in FAILFAST mode:", failfast_df.count())
#Compare all three modes
print("========== MALFORMED DATA MODE COMPARISON ==========")

print("PERMISSIVE:", permissive_df.count())
print("DROPMALFORMED:", dropmalformed_df.count())
print("FAILFAST:", failfast_df.count())

print("====================================================")

Rows read in PERMISSIVE mode: 150
+--------+-----------+------------+
|order_id|customer_id|order_amount|
+--------+-----------+------------+
|1001    |101        |100.0       |
|1002    |102        |125.0       |
|1003    |103        |150.0       |
|1004    |104        |175.0       |
|1005    |105        |NULL        |
|1006    |106        |225.0       |
|1007    |107        |INVALID     |
|1008    |108        |275.0       |
|1009    |109        |300.0       |
|1010    |110        |325.0       |
+--------+-----------+------------+
only showing top 10 rows
Rows read in DROPMALFORMED mode: 150
Rows read in FAILFAST mode: 150
========== MALFORMED DATA MODE COMPARISON ==========
PERMISSIVE: 150
DROPMALFORMED: 150
FAILFAST: 150
